In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
oss_demo = pd.read_csv('../data/Raw/RCS_Households_Demographics.csv')

In [3]:
oss_demo.head()

,Household_Index,Date_Added,Housing_Type,Household_Type,Household_Size,Annual_Income,ZIP_Code,ObjectId
0,6096,2018/09/11 23:41:00+00,Own,Single Person,1,"$13,152.00",40214.0,1
1,10233,2018/09/24 21:28:00+00,Rent/Non-Subsidized,Single Parent Female,2,$0.00,40210.0,2
2,10234,2018/09/24 21:28:00+00,Own,Single Parent Female,3,"$12,456.00",40211.0,3
3,10238,2018/09/24 21:30:00+00,Own,Single Parent Female,1,"$10,356.00",40299.0,4
4,10239,2018/09/24 21:30:00+00,Rent/Subsidized,Single Parent Female,2,"$20,672.52",40216.0,5


## Cleaning the Data

The cleaning and EDA for this dataset was completed here, and visualizations using this data were done in the database notebook once the sqlite database was completed.
- Dropped 'Household_Index' and 'Housing_Type' columns since they were unnecessary for answering the data questions. 
- Renamed leftover columns to match syntax used in other notebooks and sqlite.
- Dropped rows where the 'zipcode' == NA, since the analysis is focusing on zipcodes in Jefferson County, KY.
- Left rows where the 'date_added' == NA alone for now. May handle them differently in future data questions.
- Removed duplicated rows.
- Changed 'data_added' column dtype to datetime, 'annual_income' column to float, 'zipcode' column to string to ensure data integrity when converted to sqlite. 

In [4]:
oss_demo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50553 entries, 0 to 50552
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Household_Index  50553 non-null  object 
 1   Date_Added       47525 non-null  object 
 2   Housing_Type     49409 non-null  object 
 3   Household_Type   50553 non-null  object 
 4   Household_Size   50553 non-null  int64  
 5   Annual_Income    47677 non-null  object 
 6   ZIP_Code         49613 non-null  float64
 7   ObjectId         50553 non-null  int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 3.1+ MB


In [5]:
oss_demo = oss_demo.drop(['Household_Index', 'Housing_Type'], axis=1)

In [6]:
oss_demo = oss_demo.rename(columns={'Date_Added': 'date_added', 'Household_Type':'household_type', 'Household_Size':'household_size', 'Annual_Income':'annual_income', 'ZIP_Code':'zipcode', 'ObjectId':'household_id'})

In [7]:
oss_demo.columns

Index(['date_added', 'household_type', 'household_size', 'annual_income',
       'zipcode', 'household_id'],
      dtype='object')

In [8]:
oss_demo.isna().sum()

date_added        3028
household_type       0
household_size       0
annual_income     2876
zipcode            940
household_id         0
dtype: int64

In [9]:
oss_demo = oss_demo.dropna(subset=['zipcode'])

In [10]:
oss_demo.isna().sum()

date_added        3028
household_type       0
household_size       0
annual_income     2101
zipcode              0
household_id         0
dtype: int64

In [11]:
duplicated_mask = oss_demo.duplicated(keep=False)
duplicated_rows = oss_demo[duplicated_mask]
duplicated_rows

,date_added,household_type,household_size,annual_income,zipcode,household_id


In [12]:
oss_demo['date_added'] = pd.to_datetime(oss_demo['date_added'])

In [13]:
oss_demo['annual_income'] = oss_demo['annual_income'].replace(r'[\$,]', '', regex=True).astype(float)

In [14]:
#Removed decimal from float by first changing type to 'Int64'
oss_demo['zipcode'] = oss_demo['zipcode'].astype('Int64').astype(str)

In [15]:
oss_demo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49613 entries, 0 to 50552
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   date_added      46585 non-null  datetime64[ns, UTC]
 1   household_type  49613 non-null  object             
 2   household_size  49613 non-null  int64              
 3   annual_income   47512 non-null  float64            
 4   zipcode         49613 non-null  object             
 5   household_id    49613 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(1), int64(2), object(2)
memory usage: 2.6+ MB


## EDA

- The oss_demo dataframe included zipcodes that were unique to a business/specific area, outside of Jefferson County, KY, or were nonexistent. A filtered dataframe was created to only include zipcodes in Jefferson County, KY since that was the focus of the analysis between the Louisville Free Public Library and OSS Households.

In [16]:
oss_demo['zipcode'].unique()

array(['40214', '40210', '40211', '40299', '40216', '40212', '40229',
       '40208', '40272', '40243', '40217', '40215', '40209', '40219',
       '40241', '40118', '40203', '40206', '40258', '40218', '40223',
       '40220', '40242', '40291', '40222', '40213', '40290', '40204',
       '40202', '40207', '40245', '40205', '40228', '40252', '40059',
       '40256', '40023', '40251', '40268', '40201', '40177', '40232',
       '40200', '40269', '40047', '40250', '40280', '40150', '40257',
       '42718', '40117', '40224', '40014', '40303', '40273', '40041',
       '40165', '40259', '40227', '40231', '40253', '40226', '40124',
       '40020', '40221', '42017', '47150', '42011', '42010', '40031',
       '40528', '40056', '40018', '40292', '40416', '40114', '47130',
       '40144', '42016', '40160', '40065', '40271', '40042', '42301',
       '40289', '40263', '40109', '42101', '40155', '47129', '40071'],
      dtype=object)

In [17]:
jefferson_co_zips = ['40203', '40210', '40059', '40220', '40041', '40047', '40242',
       '40211', '40215', '40205', '40218', '40222', '40223', '40067',
       '40177', '40212', '40214', '40299', '40245', '40243', '40025',
       '40109', '40241', '40206', '40219', '40258', '40213', '40204',
       '40208', '40216', '40118', '40202', '40217', '40207', '40225',
       '40229', '40023', '40272', '40209', '40228', '40291']

In [18]:
mask = oss_demo['zipcode'].isin(jefferson_co_zips)

In [19]:
oss_demo_filtered = oss_demo[mask]

In [20]:
oss_demo_filtered.to_csv('../data/Clean/clean_oss.csv', index=False)